# Rotativo + control de volatilidad → FTMO (simulador v4)

**Idea.** El libro rotativo ya tiene ventaja (mucho CAGR) pero demasiado riesgo para una prop. Aquí se toma su **serie diaria de retornos** y se le aplica una capa de tamaño encima, de tres formas, antes de pasarla por las evaluaciones FTMO:

| Método | Qué hace |
|---|---|
| **A · Escala fija k** | Se opera el rotativo a una fracción fija k (0,25 … 1,0). Divide retorno y drawdown por igual |
| **B · Vol targeting de cartera (Carver a nivel de cartera)** | k cambia cada día = τ / volatilidad reciente del rotativo (EWMA), con tope. Menos tamaño cuando la cartera se pone nerviosa |
| **C · B + frenos de prop** | Además: freno por distancia al −10 % y colchón para que un mal día (2,5 sigmas) no toque el −5 % diario |

No toca la selección ni el timing del rotativo: solo el tamaño total. (Es distinto de la prueba de la Guía 17.4, donde Carver cambiaba el tamaño de cada activo dentro de la rotación).

**Entrada:** un CSV del rotativo con `date` y la columna de retorno diario neto de la cartera. **Salida:** carpeta nueva en `resultados/` con `resumen_rot.txt`.

## 1 · Configuración

In [ ]:
%matplotlib inline
import os, time, datetime as dt
import numpy as np, pandas as pd, matplotlib.pyplot as plt

CSV_ROTATIVO = "rotativo_daily.csv"   # <-- tu CSV diario del rotativo (date + retorno neto de la cartera)
COL_RET      = None                   # None = autodetectar (strategy_return, portfolio_return, ret, net...)
ANN          = 365                    # días por año de la serie (365 si el rotativo va en calendario de 7 días; 252 si L-V)
EVAL_START   = "2018-01-01"
IS_FRAC      = 0.70

# reglas FTMO 2-Step Swing
OBJ1, OBJ2, DD_DIA, DD_MAX, DIAS_MIN = 0.10, 0.05, 0.05, 0.10, 4
HORIZONTE  = None      # None = sin límite (como FTMO); un número = días máximos por fase
MIN_FUTURO = 252       # solo se empiezan evaluaciones con al menos este número de días por delante
PASO       = 3         # una evaluación cada 3 días
FACTOR_INTRADIA = 1.5  # peor momento del día = 1,5 x la pérdida al cierre (no hay OHLC de la cartera)
REF_DIAS   = 3         # el límite diario se mide desde el máximo equity de los últimos N días (aprox. del "saldo")
COSTE_EXTRA_ANUAL = 0.0   # coste CFD adicional por año sobre la exposición (swap); 0 = el CSV ya es neto

# rejillas
KS     = [0.25, 0.35, 0.50, 0.65, 0.80, 1.00]          # A: escala fija
TAUS   = [0.08, 0.10, 0.12, 0.15, 0.20, 0.25]          # B y C: volatilidad objetivo anual de la cartera
K_MAX  = 1.0           # tope de B/C: nunca más que el rotativo a tamaño completo (sin apalancar)
SPAN_VOL = 30
FRENO_DD_MIN, COLCHON_K, COLCHON_MIN = 0.25, 2.5, 0.01
# cuenta fondeada
DIAS_FOND, CADA_COBRO, REPARTO, CUOTA = 365, 30, 0.80, 0.0054
RIESGO_MAX_FONDEO = 0.05

RUN_DIR = os.path.join("resultados", f"rot_{dt.datetime.now():%Y%m%d_%H%M%S}"); os.makedirs(RUN_DIR, exist_ok=True)
LOG = []
def log(s=""): print(s); LOG.append(str(s))

## 2 · Datos del rotativo

In [ ]:
def cargar(path):
    if not os.path.exists(path):
        log(f"[DATOS SINTÉTICOS] no encuentro {path}: genero una serie de prueba (los números no valen)")
        rng = np.random.default_rng(1); n = 3300
        idx = pd.date_range("2017-06-01", periods=n, freq="D")
        reg = (np.sin(np.arange(n) / 150) > 0).astype(float)
        r = 0.0035 * reg - 0.0005 + rng.standard_t(4, n) * 0.012 * (0.6 + reg)
        return pd.Series(r, index=idx), False
    df = pd.read_csv(path)
    dc = [c for c in df.columns if c.lower() in ("date", "fecha", "time", "timestamp")][0]
    df[dc] = pd.to_datetime(df[dc]); df = df.set_index(dc).sort_index()
    col = COL_RET or next(c for c in ["strategy_return", "portfolio_return", "port_ret", "net", "ret", "return", "retorno"] if c in df.columns)
    log(f"CSV: {path} | columna de retorno: {col}")
    return df[col].astype(float).fillna(0.0), True

r, REAL = cargar(CSV_ROTATIVO)
r = r.loc[pd.Timestamp(EVAL_START) - pd.Timedelta(days=120):]       # un poco antes para la vol
idx_eval = r.loc[EVAL_START:].index
IS_END = idx_eval[int(len(idx_eval) * IS_FRAC) - 1]
log(f"Serie {r.index[0].date()} -> {r.index[-1].date()} ({len(r)} días) | IS hasta {IS_END.date()}")

def met(x):
    x = x.dropna(); eq = (1 + x).cumprod(); yrs = len(x) / ANN; sd = x.std()
    return {"Sharpe": x.mean() / sd * np.sqrt(ANN), "CAGR %": (eq.iloc[-1] ** (1 / yrs) - 1) * 100,
            "MaxDD %": (eq / eq.cummax() - 1).min() * 100, "Vol %": sd * np.sqrt(ANN) * 100,
            "Peor día %": x.min() * 100, "Días<-3% (%)": (x < -0.03).mean() * 100, "Días<-5% (%)": (x < -0.05).mean() * 100}
re = r.loc[EVAL_START:]
log("\n=== El rotativo tal cual ===")
log(pd.DataFrame({"Total": met(re), "IS": met(re.loc[:IS_END]), "OOS": met(re.loc[IS_END:].iloc[1:])}).T.round(2).to_string())
log(f"Los 8 peores días: {(re.nsmallest(8) * 100).round(2).to_dict()}")

## 3 · Las tres capas de tamaño

`k_t` = fracción del rotativo que se opera el día t, decidida con datos hasta t−1. La vol de la cartera es EWMA (span 30) de sus propios retornos.

In [ ]:
vol_ewm = (r.ewm(span=SPAN_VOL, adjust=False).std() * np.sqrt(ANN)).shift(1).bfill()

def serie_k(metodo, p):
    if metodo == "A": return pd.Series(p, index=r.index)
    return (p / vol_ewm).clip(upper=K_MAX).fillna(0)

def backtest(metodo, p):
    k = serie_k(metodo, p)
    out = k * r - k * COSTE_EXTRA_ANUAL / ANN
    return out, k

rows = {}
for m, grid in [("A", KS), ("B", TAUS)]:
    for p in grid:
        x, k = backtest(m, p); x = x.loc[EVAL_START:]
        d = met(x); d["k medio"] = k.loc[EVAL_START:].mean(); rows[f"{m} {p}"] = d
TB = pd.DataFrame(rows).T.round(2)
log("\n=== Backtest continuo con cada capa (sin reglas de prop) ===")
log(TB.to_string())

## 4 · Simulador FTMO sobre retornos diarios

Sin precios de cada activo, el simulador trabaja con el equity de la cartera:

- **Límite diario:** la pérdida del día × `FACTOR_INTRADIA` (peor momento del día) no puede pasar del 5 % de la cuenta, medida desde el **máximo equity de los últimos `REF_DIAS` días**. Así se aproxima, del lado prudente, que FTMO mide desde el saldo y que una pérdida flotante arrastrada cuenta.
- **Límite máximo:** −10 % estático (también con el factor intradía).
- **C (frenos):** freno por distancia al −10 % y colchón: la k se reduce para que 2,5 sigmas de la cartera dejen al menos un 1 % hasta el límite diario.

In [ ]:
rv = r.values; volv = vol_ewm.values; T = len(rv); dates = r.index

def simular(metodo, p, i0, objetivo, dias_max, cobros=False):
    K = 1.0; E = 1.0; hist = [1.0]; dias = 0; cobrado = 0.0; ult = 0
    for d in range(1, dias_max + 1):
        t = i0 + d
        if t >= T: return "fin_datos", d, cobrado
        k = p if metodo == "A" else min(K_MAX, p / volv[t]) if volv[t] > 0 else 0.0
        ref = max(hist[-REF_DIAS:])
        if metodo == "C":
            k *= min(1.0, max(FRENO_DD_MIN, (E - (1 - DD_MAX)) / DD_MAX))
            hueco = E - (ref - DD_DIA); peor = COLCHON_K * k * volv[t] / np.sqrt(ANN) * E
            if peor > 0 and hueco - COLCHON_MIN < peor: k *= max(0.0, (hueco - COLCHON_MIN) / peor)
        x = k * rv[t] - k * COSTE_EXTRA_ANUAL / ANN
        peor_dia = E * (1 + min(0.0, x) * FACTOR_INTRADIA)
        E = E * (1 + x); hist.append(E)
        if k > 0: dias += 1
        if peor_dia < ref - DD_DIA: return "rotura_diaria", d, cobrado
        if peor_dia < 1 - DD_MAX:   return "rotura_maxima", d, cobrado
        if not cobros and E >= 1 + objetivo and dias >= DIAS_MIN: return "pasa", d, cobrado
        if cobros and d - ult >= CADA_COBRO and E > 1:
            cobrado += REPARTO * (E - 1); E = 1.0; hist = [1.0]; ult = d
    return ("sin_resolver" if not cobros else "sobrevive"), dias_max, cobrado

def evaluacion(m, p, i0):
    h1 = HORIZONTE or (T - i0 - 1)
    a, d1, _ = simular(m, p, i0, OBJ1, h1)
    if a != "pasa": return a, d1, 1
    h2 = HORIZONTE or (T - i0 - d1 - 1)
    b, d2, _ = simular(m, p, i0 + d1, OBJ2, h2)
    return b, d1 + d2, 2

t0 = dates.get_loc(idx_eval[0]); t_is = dates.get_loc(IS_END)
INI = list(range(t0, T - MIN_FUTURO, PASO))
INI_F = list(range(t0, T - DIAS_FOND, PASO * 3))
GRID = [("A", k) for k in KS] + [("B", t) for t in TAUS] + [("C", t) for t in TAUS]
log(f"\nEvaluaciones por combinación: {len(INI)} | cuentas fondeadas: {len(INI_F)}")

## 5 · Evaluaciones (fase 1 + fase 2)

In [ ]:
tt = time.time(); res = []
for m, p in GRID:
    for i0 in INI:
        a, d, f = evaluacion(m, p, i0)
        res.append({"metodo": m, "param": p, "inicio": dates[i0], "tramo": "IS" if i0 <= t_is else "OOS", "resultado": a, "dias": d, "fase": f})
RE = pd.DataFrame(res); log(f"({time.time()-tt:.0f} s)")

def resumen(df):
    g = df.groupby(["metodo", "param"])
    return pd.DataFrame({
        "Pasa %": g.apply(lambda x: (x.resultado == "pasa").mean() * 100),
        "Rot.diaria %": g.apply(lambda x: (x.resultado == "rotura_diaria").mean() * 100),
        "Rot.máx %": g.apply(lambda x: (x.resultado == "rotura_maxima").mean() * 100),
        "Sin resolver %": g.apply(lambda x: x.resultado.isin(["sin_resolver", "fin_datos"]).mean() * 100),
        "Días mediana": g.apply(lambda x: x.loc[x.resultado == "pasa", "dias"].median()),
        "Días p75": g.apply(lambda x: x.loc[x.resultado == "pasa", "dias"].quantile(0.75)),
        "Días p90": g.apply(lambda x: x.loc[x.resultado == "pasa", "dias"].quantile(0.90)),
    }).round(1)
for tr in ["IS", "OOS"]:
    log(f"\n=== Evaluación — inicios en {tr} ==="); log(resumen(RE[RE.tramo == tr]).to_string())
TOT = resumen(RE); log("\n=== Evaluación — todos ==="); log(TOT.to_string()); TOT.to_csv(os.path.join(RUN_DIR, "evaluacion_resumen.csv"))

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
for m, grid in [("A", KS), ("B", TAUS), ("C", TAUS)]:
    s = TOT.loc[m]; ax[0].plot(range(len(grid)), s["Pasa %"].values, marker="o", label=m); ax[1].plot(range(len(grid)), s["Días mediana"].values, marker="o", label=m)
ax[0].set_title("% que pasa las dos fases"); ax[1].set_title("días hasta pasar (mediana)")
for a_ in ax: a_.set_xlabel("parámetro de menor a mayor (k o τ, ver tabla)"); a_.legend(); a_.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(os.path.join(RUN_DIR, "evaluacion.png"), dpi=110); plt.show()

## 6 · Cuenta fondeada (12 meses, cobro cada 30 días)

In [ ]:
tt = time.time(); rf = []
for m, p in GRID:
    for i0 in INI_F:
        a, d, c = simular(m, p, i0, 0, DIAS_FOND, cobros=True)
        rf.append({"metodo": m, "param": p, "tramo": "IS" if i0 <= t_is else "OOS", "resultado": a, "cobrado": c})
RF = pd.DataFrame(rf); log(f"({time.time()-tt:.0f} s)")
g = RF.groupby(["metodo", "param"])
FO = pd.DataFrame({"Pierde %": g.apply(lambda x: (x.resultado != "sobrevive").mean() * 100),
                   "Cobrado medio % cuenta": g.apply(lambda x: x.cobrado.mean() * 100),
                   "Cobrado mediana %": g.apply(lambda x: x.cobrado.median() * 100)}).round(2)
log("\n=== Cuenta fondeada ==="); log(FO.to_string()); FO.to_csv(os.path.join(RUN_DIR, "fondeada_resumen.csv"))

# mejores marchas: evaluación con más % de pasar entre las que pasan en mediana <= 120 días; fondeada con pierde <= 5 %
cand = TOT[(TOT["Días mediana"] <= 120)].sort_values("Pasa %", ascending=False).head(5)
log("\n=== Mejores para la evaluación (mediana <= 120 días) ==="); log(cand.to_string())
fcand = FO[FO["Pierde %"] <= RIESGO_MAX_FONDEO * 100].sort_values("Cobrado medio % cuenta", ascending=False).head(5)
log("\n=== Mejores para la cuenta fondeada (pierde <= 5 %) ==="); log(fcand.to_string())
if not REAL: log("\nAVISO: DATOS SINTÉTICOS. Los números no valen.")
open(os.path.join(RUN_DIR, "resumen_rot.txt"), "w", encoding="utf-8").write("\n".join(LOG))
RE.to_csv(os.path.join(RUN_DIR, "evaluaciones_detalle.csv"), index=False)
print("Guardado en", RUN_DIR)